# IOAI — 2025 Residential Camp Cv Gan (Colab 자동 설정판)

이 노트북은 IOAI 로컬 연습 사이트에서 **데이터·학습환경이 자동 준비**되도록 생성되었습니다.
아래 **설정 셀을 먼저 실행**하면 공식 GitHub 저장소에서 이 문제 폴더만 부분 클론으로 받아
(전체 6.6GB 가 아니라 해당 폴더만), 그 폴더로 이동한 뒤 이후 셀이 그대로 학습/예측을 합니다.
완료 후 생성되는 제출 파일을 내려받아 연습 사이트의 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** 로 바꾸면 학습이 빨라집니다.

In [ ]:
# === 데이터 + 환경 자동 설정 (가장 먼저 실행) ===
# 공식 공개 저장소에서 이 문제 폴더만 부분 클론(sparse)으로 받고 그 폴더로 이동한다.
import os
REPO_URL = "https://github.com/teodora-dimitrova/Singapore-Team-Selection"
CLONE = "Singapore-Team-Selection"
SUBDIR = "IOAI 2025 Residential camp task/ 2025 Camp Test CV"
WORKDIR = "IOAI 2025 Residential camp task/ 2025 Camp Test CV"
# Colab 은 /content 가 홈. 재실행해도 경로가 안정적이도록 고정 기준에서 시작한다.
BASE = "/content" if os.path.isdir("/content") else os.getcwd()
os.chdir(BASE)
if not os.path.isdir(os.path.join(CLONE, SUBDIR)):
    !git clone --filter=blob:none --no-checkout --depth 1 $REPO_URL $CLONE
    !cd $CLONE && git sparse-checkout set "$SUBDIR"
    !cd $CLONE && git checkout
os.chdir(os.path.join(BASE, CLONE, WORKDIR))
print("작업 폴더:", os.getcwd())
print("내용:", sorted(os.listdir(".")))

# DCGAN Image Generation (MNIST)

> **Singapore IOAI 2025 — Residential Camp (CV, Part 1)**

Train a **DCGAN** to generate MNIST digits. Generation quality is scored **objectively by FID (Fréchet Inception Distance)** — the standard GAN metric, **lower is better** — computed against real MNIST in a bundled feature space. **Submit** `submission.zip` (`submission_model.py` with `class Generator` + `submission_dic.pth` = generator weights); the grader generates 4096 images and measures FID. Improve the architecture/epochs to lower it.

(Part 2 — StyleGAN latent editing — and the full reference answers are in the **Solution** tab.)

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, zipfile
import torchvision, torchvision.transforms as T
from torch.utils.data import DataLoader
torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
LATENT_DIM = 100
device

## Data — MNIST in [-1, 1] (matches the generator's Tanh output)

In [ ]:
tf = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])
train = torchvision.datasets.MNIST('./data', train=True, download=True, transform=tf)
loader = DataLoader(train, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
len(train)

## DCGAN Generator + Discriminator

In [ ]:
import torch
import torch.nn as nn


class Generator(nn.Module):
    def __init__(self, latent_dim=100, ngf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),                 # 4x4
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 3, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),                 # 7x7
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),                     # 14x14
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False),
            nn.Tanh(),                                              # 28x28
        )

    def forward(self, z):
        return self.main(z.view(z.size(0), -1, 1, 1))

class Discriminator(nn.Module):
    def __init__(self, ndf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False), nn.LeakyReLU(0.2, True),          # 14x14
            nn.Conv2d(ndf, ndf * 2, 4, 2, 1, bias=False), nn.BatchNorm2d(ndf * 2), nn.LeakyReLU(0.2, True),  # 7x7
            nn.Conv2d(ndf * 2, ndf * 4, 3, 2, 1, bias=False), nn.BatchNorm2d(ndf * 4), nn.LeakyReLU(0.2, True),  # 4x4
            nn.Conv2d(ndf * 4, 1, 4, 1, 0, bias=False),                                 # 1x1 logit
        )
    def forward(self, x):
        return self.main(x).view(-1)

G = Generator(LATENT_DIM).to(device)
D = Discriminator().to(device)
crit = nn.BCEWithLogitsLoss()
optG = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
optD = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

## Train (a few epochs is enough for a baseline)

In [ ]:
EPOCHS = 5
for epoch in range(EPOCHS):
    for real, _ in loader:
        real = real.to(device); n = real.size(0)
        # --- D ---
        z = torch.randn(n, LATENT_DIM, device=device)
        fake = G(z)
        lossD = crit(D(real), torch.ones(n, device=device)) + \
                crit(D(fake.detach()), torch.zeros(n, device=device))
        optD.zero_grad(); lossD.backward(); optD.step()
        # --- G ---
        lossG = crit(D(fake), torch.ones(n, device=device))
        optG.zero_grad(); lossG.backward(); optG.step()
    print(f'epoch {epoch+1}/{EPOCHS}  lossD {lossD.item():.3f}  lossG {lossG.item():.3f}')

## Save submission (`submission.zip` = `submission_model.py` + `submission_dic.pth`)

In [ ]:
torch.save(G.state_dict(), 'submission_dic.pth')
with open('submission_model.py', 'w') as f:
    f.write('''import torch
import torch.nn as nn


class Generator(nn.Module):
    def __init__(self, latent_dim=100, ngf=64):
        super().__init__()
        self.main = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, ngf * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(ngf * 4), nn.ReLU(True),                 # 4x4
            nn.ConvTranspose2d(ngf * 4, ngf * 2, 3, 2, 1, bias=False),
            nn.BatchNorm2d(ngf * 2), nn.ReLU(True),                 # 7x7
            nn.ConvTranspose2d(ngf * 2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf), nn.ReLU(True),                     # 14x14
            nn.ConvTranspose2d(ngf, 1, 4, 2, 1, bias=False),
            nn.Tanh(),                                              # 28x28
        )

    def forward(self, z):
        return self.main(z.view(z.size(0), -1, 1, 1))
''')
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('submission_model.py'); z.write('submission_dic.pth')
print('wrote submission.zip')

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.zip']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)